# Session 3 — Introduction to LangChain

Last session we called **three providers** from Python and got structured data back. It worked —
but supporting OpenAI *and* Gemini *and* Anthropic meant writing everything **three times**: three
SDKs, three ways to send a system prompt, three ways to read the text, three token-usage fields.

Today we fix that with **LangChain**: *one* interface across providers, plus the building blocks
(models, messages, tools, memory, structured output) we'll lean on for the rest of the course.

**Our client's ask (Northstar's CTO):** *"I don't want to be locked to one model vendor, and I want
code my team can maintain."* By the end, swapping providers will be a **one-line change**.

**What we'll do**
1. Why a framework — and how **LangChain / LangGraph / LangSmith** divide the work
2. The **standard model interface** (`init_chat_model`) + the **provider swap**
3. **Messages** and **structured output** — the S2 techniques, now once instead of three times
4. A **first agent** with a **tool** (`create_agent`)
5. A taste of **memory** and **streaming**
6. How to **read the LangChain docs**

## 1. Setup

🎯 **Purpose:** install LangChain v1 + the Google integration, and load our API key.

> 💡 **Real-life:** LangChain is a **universal travel adapter**. Every country (provider) has a
> different socket, but you carry **one adapter** and your laptop just plugs in — you don't rewire
> your charger per country, you swap the little plug.

We install `langchain` with the `google-genai` extra (that pulls in the Gemini integration),
`langgraph` (for the memory checkpointer later), and the usual helpers.

In [ ]:
%pip install -q -U "langchain[google-genai]" langgraph python-dotenv pydantic pyyaml
# To try the provider-swap on other vendors, also:  %pip install -q -U "langchain[anthropic]" "langchain[openai]"

In [ ]:
from dotenv import load_dotenv
load_dotenv()  # reads .env — LangChain's Google integration accepts GEMINI_API_KEY (from S2) OR GOOGLE_API_KEY

# Quick key check (prints set/MISSING — never the key value itself)
import os
for k in ["GEMINI_API_KEY", "GOOGLE_API_KEY", "ANTHROPIC_API_KEY", "OPENAI_API_KEY"]:
    print(f"{k:20s}", "set" if os.environ.get(k) else "MISSING")

## 2. Why a framework? And who's who: LangChain vs LangGraph vs LangSmith

A framework is a **trade**. You *gain* one interface across providers, reusable building blocks,
and a straight path to agents/RAG/tracing later. You *pay* a dependency and a little abstraction.
The rule: **use it when the shared interface saves you more than the abstraction costs.** For a
one-off script, raw SDK is fine. For an app that grows every week — like ours — a framework earns
its keep.

Three names people constantly confuse — three different jobs:

| Name | Job | We use it… |
|---|---|---|
| **LangChain** | The **building blocks & standard interface** — models, messages, tools, structured output. *"One API for any model."* | today |
| **LangGraph** | The **control-flow / workflow engine** — state, nodes, edges, routing, human-in-the-loop. Runs *underneath* agents. | Session 6 (deep) |
| **LangSmith** | The **observability layer** — tracing, debugging, evaluation. It *watches* your runs. | Week 2 |

They nest: LangChain's `create_agent` is **built on** LangGraph; LangSmith sits beside both, recording.

> 💡 **Real-life (a kitchen):** **LangChain** = standard appliances & ingredients (any brand fits the
> same sockets). **LangGraph** = the **recipe** that decides which step happens when, and what to do
> if a step fails. **LangSmith** = the **CCTV + food-safety inspector** reviewing the tape afterwards.
> You can cook with just the appliances; the recipe and the camera matter more as the meal gets complex.

**Do you need all three today? No** — just LangChain, with one peek at LangGraph (memory).

## 3. The standard model interface

🎯 **Purpose:** create a model with `init_chat_model` and make a first call — the whole pitch in a
few lines.

The provider lives *inside the model string*: `"<provider>:<model>"`. `init_chat_model` reads the
API key from the environment and hands back a model object with a uniform interface:
`.invoke(...)` → an **`AIMessage`** whose `.text` is the reply and whose `.usage_metadata` is the
token count (same field names for **every** provider — hold that thought).

In [ ]:
from langchain.chat_models import init_chat_model

MODEL = "google_genai:gemini-3.1-flash-lite"   # "<provider>:<model>" — this string is the swap point
model = init_chat_model(MODEL)

resp = model.invoke("Say hello to an AI engineering class in one sentence.")
print(resp.text)

### The payoff: swap the provider by changing ONE string

🎯 **Purpose:** this is exactly what the CTO asked for. In Session 2, supporting three vendors meant
three code paths. Here it's a **list of strings** — same `.invoke`, same `.text`, same everything.

> 💡 **Real-life:** swapping the model is **changing the engine without rebuilding the car** — same
> dashboard and pedals, different engine under the hood.

(If you only have a Google key, the others are skipped gracefully — set that provider's key to try it.)

In [ ]:
for model_id in ["google_genai:gemini-3.1-flash-lite",
                 "anthropic:claude-haiku-4-5",
                 "openai:gpt-5.4-mini"]:
    try:
        m = init_chat_model(model_id)
        print(model_id, "->", m.invoke("Reply with exactly one word: hello").text)
    except Exception as e:
        print(model_id, "-> skipped:", type(e).__name__, "(install that integration / set its API key to try it)")

## 4. Messages & prompts

🎯 **Purpose:** use LangChain's typed message objects — identical across providers. No more "system
goes in `instructions` for OpenAI but `system=` for Anthropic but `config` for Gemini."

Same mental model as Session 2 — a conversation of **roles**: system, human (user), AI (assistant).

> 💡 **Real-life:** the roles are the **theatre script** — `system` is the director's standing notes,
> `human` is the other character's lines, `AI` is the actor's reply. LangChain gives you one script
> format every actor (provider) can read.

In [ ]:
from langchain.messages import SystemMessage, HumanMessage

SYSTEM = "You are a terse support assistant for Northstar Services. Answer in ONE sentence."
USER   = "A customer asks how to reset their password."

resp = model.invoke([SystemMessage(SYSTEM), HumanMessage(USER)])
print(resp.text)

The same thing with plain dicts (the OpenAI-style format from Session 2 still works). Use whichever
you prefer — the typed objects give you autocomplete; the dicts are familiar.

In [ ]:
resp = model.invoke([
    {"role": "system", "content": SYSTEM},
    {"role": "user",   "content": USER},
])
print(resp.text)

> ⚠️ **Still stateless.** LangChain doesn't secretly remember for you here — you pass the message
> list each time, just like S2. Later (§7) we let the framework hold that list for us with *memory*.

## 5. Structured output — one API instead of three

🎯 **Purpose:** get a validated Pydantic object back with **one** method, `with_structured_output`
(replaces S2's three provider-specific parse calls).

In Session 2 structured output was our #1 technique — but each provider had its own parse call. Here
it's `model.with_structured_output(YourSchema)`, and it works the same on any provider.

> 💡 **Real-life:** it's handing the model a **fill-in-the-boxes form** and getting the *filled form*
> back instead of a letter — `Literal[...]` fields are dropdowns, `Field(description=...)` is the hint
> written next to each box.

First, the exact same `Triage` schema from Session 2:

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal

class Triage(BaseModel):
    category:  Literal["billing", "technical", "account", "general"] = Field(description="Main topic of the message")
    urgency:   Literal["low", "medium", "high"]                      = Field(description="How quickly this needs a response")
    sentiment: Literal["negative", "neutral", "positive"]            = Field(description="The customer's mood")
    summary:   str = Field(description="One-line summary of what the customer wants")

In [ ]:
MESSAGE = "I've been charged twice for May and nobody has replied to my emails. This is ridiculous."

triage_model = model.with_structured_output(Triage)   # one call, any provider
t = triage_model.invoke(MESSAGE)
print(type(t).__name__, "->", t)
print("category:", t.category, "| urgency:", t.urgency)   # a real, validated object

### The one catch: tokens

🎯 **Purpose:** when you use `with_structured_output`, `.invoke` returns the **parsed object**, not
the `AIMessage` — so you lose `.usage_metadata`. If you still want token counts, ask for the raw
message too with `include_raw=True`. You get back a dict: `{"parsed", "raw", "parsing_error"}`.

> 💡 **Real-life:** `include_raw=True` also keeps the **envelope** the form arrived in — that's where
> the token 'receipt' (usage) lives.

In [ ]:
triage_raw = model.with_structured_output(Triage, include_raw=True)
out = triage_raw.invoke(MESSAGE)

print("keys:", sorted(out.keys()))
print("parsed :", out["parsed"].category, out["parsed"].urgency)   # the Triage object
print("tokens :", out["raw"].usage_metadata)                       # same keys on every provider
print("error  :", out["parsing_error"])                            # None when all good

## 6. Tools & a first agent

🎯 **Purpose:** stop *us* driving the model. Give it **tools** (plain Python functions) and let *it*
decide which to call. This is the heart of the next few sessions — here's the simplest version.

A **tool** is just a function with a **docstring**. The docstring is the model's instruction manual:
it reads the name, arguments, and description to decide *when* to call it. Write docstrings like
you're onboarding an intern.

> 💡 **Real-life:** an agent is a **new hire with a phone directory**. You don't script every call —
> you say "sort this out," and they decide who to phone (which tool), listen, and report back. The
> docstrings are the labels in the directory.

In [ ]:
from langchain.agents import create_agent

def get_order_status(order_id: str) -> str:
    """Look up the delivery status of a customer order by its ID."""
    # fake data for the demo — Session 4 makes this hit real data
    return f"Order {order_id}: shipped, arriving tomorrow."

def refund_policy() -> str:
    """Return Northstar's refund policy in one sentence."""
    return "Refunds are available within 30 days of purchase for unused subscriptions."

agent = create_agent(
    model=MODEL,                                  # same "google_genai:..." string
    tools=[get_order_status, refund_policy],
    system_prompt="You are a Northstar Services support assistant. Use tools when they help.",
)

result = agent.invoke({"messages": [
    {"role": "user", "content": "Where is order A-1007, and what's your refund policy?"}
]})
print(result["messages"][-1].text)   # the final answer, after the agent used the tools

We never told it *which* tool to call — it read the question, matched the tool descriptions, called
both, and wrote one answer. `result["messages"]` is the **whole transcript**; `[-1]` is the final
reply. Let's see the steps in between:

In [ ]:
for step in result["messages"]:
    step.pretty_print()   # user message -> the model's tool calls -> tool results -> final reply

## 7. Memory & streaming — a short taste

🎯 **Purpose:** two things that make an assistant feel real — remembering the conversation, and
replying as it types.

**Memory.** The model is stateless (§4). In S2 *we* resent the history. An agent can hold it for us —
give it a **checkpointer** and a **thread_id**, and it files each conversation like a support ticket.

> 💡 **Real-life:** the `thread_id` is the **support-ticket number**. Everyone who reopens ticket #42
> sees the whole thread; ticket #43 is a stranger.

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

mem_agent = create_agent(model=MODEL, tools=[], checkpointer=InMemorySaver())

cfg = {"configurable": {"thread_id": "customer-42"}}   # the "ticket number"
print(mem_agent.invoke({"messages": [{"role": "user", "content": "Hi, my name is Priya."}]}, cfg)["messages"][-1].text)
print(mem_agent.invoke({"messages": [{"role": "user", "content": "What's my name?"}]},        cfg)["messages"][-1].text)
# It remembers "Priya" because both calls share thread_id "customer-42". Change the id -> a stranger.

**Streaming.** Same `model`, but `.stream` yields **chunks** as the model types, instead of waiting
for the whole reply. This is what makes a chat UI feel alive — we use it in the Streamlit app (S7).

> 💡 **Real-life:** streaming is watching the reply **typed live** vs. waiting for the sealed letter.

In [ ]:
for chunk in model.stream("Write a two-sentence apology to a customer who was double-charged."):
    print(chunk.text, end="", flush=True)

## 8. Working with the LangChain docs

LangChain moves fast and has a big surface. The skill that outlasts any one API is **finding the
right building block**. The loop:

> **find its page → read the *Python* tab → copy the smallest example → run it → adapt.**

- Use the **v1** docs and the **Python** tab (JS examples look similar but aren't identical).
- The five pages that cover ~90% of what we do: **Models**, **Messages**, **Structured output**,
  **Tools**, **Agents** (+ **Short-term memory**, **Streaming**).
- A page's "initialize a model" tabs show every provider side by side — that layout *is* the
  "one interface" idea.
- If you find an old blog post using `create_react_agent`, check the **migration guide** — v1 renamed
  it to `create_agent`.

## 9. Recap — what changed from Session 2

| Task | Session 2 (raw SDKs) | Session 3 (LangChain v1) |
|---|---|---|
| pick a model | 3 different clients | `init_chat_model("provider:model")` |
| **swap provider** | rewrite the call | **change one string** |
| system prompt | `system=` / `instructions=` / `config=` | `SystemMessage(...)` in the list |
| basic call | `.create` / `.generate_content` / `.responses` | `model.invoke(...)` → `.text` |
| structured output | 3 different `.parse` APIs | `model.with_structured_output(Schema)` |
| token usage | 3 different field names | `msg.usage_metadata["input_tokens"]` (uniform) |
| tools / agent | hand-rolled | `create_agent(model, tools, system_prompt)` |
| memory | resend history yourself | `checkpointer=InMemorySaver()` + `thread_id` |

**We gave the CTO what they asked for:** same features, vendor-agnostic, tidier. We also met the
building blocks the rest of the course uses.

**But notice** our assistant still *guesses* about accounts — ask it "where's my refund?" and it makes
something up. **Next session (S4)** we give it **real tools** to look up **real data**, turn it into a
proper **agent** with **guardrails**, and switch on **LangSmith** to watch it work.

> The project file `main.py` graduated **v0 → v1** on LangChain this session — open it alongside the
> Session 2 version to see how much provider-specific code *disappeared*.